In [1]:
# Author: Arthur Prigent
# Email: arthur.prigent@univ-brest.fr

In [1]:
import matplotlib.patches as mpatches
from scipy.stats import pearsonr
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy
import matplotlib
import scipy
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import matplotlib.colors as mcolors

from datetime import datetime, timedelta
import cartopy.feature as cfeature
import numpy as np
import xarray as xr
import glob
import matplotlib.pyplot as plt
import netCDF4
import string

import numpy as np
import xarray as xr
from scipy.spatial import cKDTree
from datetime import timedelta
import gsw
import ArcTools as Atools

# TEMP

In [2]:
# ds_all = xr.open_dataset('/data0/user/aprigent/PROCESSED/level_3/temp_common_full.nc',decode_times=False)
# # ds_all# # path = "/data0/user/aprigent/ISAS/ISAS23_DMFD_m*_TEMP.nc"
# # # # files = sorted(glob.glob(path))
# # # # temp_isas_01 = xr.open_dataset(files[0])
# # # # # Open and concatenate along a new dimension
# # # # ds_ISAS_tmp = xr.concat(
# # # #     [xr.open_dataset(f).TEMP.isel(t=0) for f in files],
# # # #     dim="month"
# # # # )


# # # # ds_ISAS = xr.Dataset({'temp':(['month','depth','latitude','longitude'],ds_ISAS_tmp.values)},
                     
# # # #                      coords={'month': np.arange(1,13,1),
# # # #                              'depth': temp_isas_01.depth.values,
# # # #                              'latitude': temp_isas_01.latitude.values,
# # # #                              'longitude': temp_isas_01.longitude.values})
# # # # ds_ISAS.to_netcdf('/data0/user/aprigent/ISAS/ISAS23_TEMP_clim.nc')
# ds_ISAS = xr.open_dataset('/data0/user/aprigent/ISAS/ISAS23_TEMP_clim.nc')
# ds_all_anom = ds_all.copy()
# # # del ds_all


# # # ds_masks = xr.open_dataset('/data0/user/aprigent/mask_regions.nc')

# # # # Interpolate basin mask to profile positions
# # # basin_at_profiles = ds_masks.interp(
# # #     lon=ds_all_anom.longitude % 360,
# # #     lat=ds_all_anom.latitude,
# # #     method="nearest"
# # # )

# # # profiles_anom_amerasian = ds_all_anom.where(
# # #     basin_at_profiles.BARENTS == 1,
# # #     drop=True
# # # )
# # # # Only profiles after 01/01/2005
# # # # profiles_anom_amerasian = profiles_anom_amerasian.where(profiles_anom_amerasian.time >= 731947, drop=True)
# # convert ordinal time to datetime
# times = ds_all_anom.time.values
# dt_list = [datetime.fromordinal(int(t)) for t in times]


# # del ds_all_psal,ds_all_psal_anom


# profile_temp = np.array(ds_all_anom.temperature_QC.values)
# lons_temp = xr.DataArray(ds_all_anom.longitude.values, dims="profile")
# lats_temp = xr.DataArray(ds_all_anom.latitude.values, dims="profile")

# months = xr.DataArray([dt.month for dt in dt_list], dims="profile")
    
    
# # # bi-linearly interpolate ISAS at profiles' locations    
# temp_isas_profiles = ds_ISAS.temp.interp(
#     longitude=lons_temp,
#     latitude=lats_temp)
# del ds_ISAS

# # # Take ISAS profiles at corresponding month
# temp_isas_profiles = temp_isas_profiles.sel(month=months)


# temp_anom = profile_temp - temp_isas_profiles.values

# ds_all_anom['temperature_QC_anom'] = (('profile','depth'),temp_anom)
# ds_all_anom['temperature_clim'] = (('profile','depth'),temp_isas_profiles.values)
# del temp_isas_profiles
# ds_all_anom['source'] = ds_all_anom['source'].astype(str)
# # ds_all_anom.to_netcdf('/data0/user/aprigent/PROCESSED/level_4/temp_anomalies_relative_ISAS_arctic.nc')

In [3]:
import xarray as xr
import numpy as np
from datetime import datetime
import gc


def compute_temp_anomalies_per_basin(
    temp_file,
    isas_file,
    mask_file,
    basin_name,
    output_file,
    start_date=None,
):
    """
    Compute temperature anomalies relative to ISAS climatology for one basin.

    Parameters
    ----------
    temp_file : str
        Path to profile dataset.
    isas_file : str
        Path to ISAS climatology dataset.
    mask_file : str
        Path to basin mask dataset.
    basin_name : str
        Name of basin variable in mask file (e.g. 'BARENTS').
    output_file : str
        Output NetCDF filename.
    start_date : int or None
        Optional ordinal date threshold (e.g. 731947 for 2005-01-01).
    """

    print(f"\nProcessing basin: {basin_name}")

    ##########################
    # Load datasets
    ##########################
    ds_all_temp = xr.open_dataset(temp_file, decode_times=False)
    ds_ISAS = xr.open_dataset(isas_file)
    ds_masks = xr.open_dataset(mask_file)

    ##########################
    # Interpolate basin mask
    ##########################
    basin_at_profiles = ds_masks.interp(
        lon=ds_all_temp.longitude % 360,
        lat=ds_all_temp.latitude,
        method="nearest"
    )

    ##########################
    # Select basin profiles
    ##########################
    profiles = ds_all_temp.where(
        basin_at_profiles[basin_name] == 1,
        drop=True
    )

    ##########################
    # Optional time filtering
    ##########################
    if start_date is not None:
        profiles = profiles.where(
            profiles.time >= start_date,
            drop=True
        )

    ##########################
    # Convert ordinal time to datetime
    ##########################
    times = profiles.time.values
    dt_list = [datetime.fromordinal(int(t)) for t in times]

    ##########################
    # Prepare coordinates
    ##########################
    profile_temp = np.array(profiles.temperature_QC.values)

    lons_temp = xr.DataArray(
        profiles.longitude.values,
        dims="profile"
    )

    lats_temp = xr.DataArray(
        profiles.latitude.values,
        dims="profile"
    )

    months = xr.DataArray(
        [dt.month for dt in dt_list],
        dims="profile"
    )
    print('Little check for alignment:')
    for i in range(100,122):
        print(
            f"Profile {i}: "
            f"time={dt_list[i]}, "
            f"month={months.values[i]}"
        )

    ##########################
    # Interpolate ISAS climatology
    ##########################
    temp_isas_profiles = ds_ISAS.temp.interp(
        longitude=lons_temp,
        latitude=lats_temp
    )

    # Select matching month
    temp_isas_profiles = temp_isas_profiles.sel(month=months)

    ##########################
    # Compute anomalies
    ##########################
    temp_anom = profile_temp - temp_isas_profiles.values

    profiles["temperature_QC_anom"] = (
        ("profile", "depth"),
        temp_anom
    )
    profiles['temperature_clim'] = (
        ('profile','depth'),
        temp_isas_profiles.values)

    # Ensure string type is NetCDF compatible
    profiles["source"] = profiles["source"].astype(str)
    profiles["month_used_for_clim"] = months

    ##########################
    # Save output
    ##########################
    profiles.to_netcdf(output_file)

    print(f"Saved: {output_file}")
    return profiles




In [4]:


temp_file = "/data0/user/aprigent/PROCESSED/level_3/temp_common_full.nc"
isas_file = "/data0/user/aprigent/ISAS/ISAS23_TEMP_clim.nc"
mask_file = "/data0/user/aprigent/mask_regions.nc"

# basins = [
#     "BARENTS",

# ]
    # "EURASIAN",
    # "AMERASIAN",
    # "NORDIC",
    # "AMERASIAN_SHELF",
    # "SIBERIAN_SHELF",
    # "BAFFIN",
    # "KARA"
    
    

basins = [
    "BARENTS",
    "EURASIAN",
    "AMERASIAN",
    "NORDIC",
    "AMERASIAN_SHELF",
    "SIBERIAN_SHELF",
    "BAFFIN",
    "KARA"
]
    
for basin in basins:

    output_file = (
        f"/data0/user/aprigent/PROCESSED/level_4/"
        f"temp_anomalies_relative_ISAS_{basin.lower()}_full_after_2005.nc"
    )

    profiles_barents = compute_temp_anomalies_per_basin(
        temp_file=temp_file,
        isas_file=isas_file,
        mask_file=mask_file,
        basin_name=basin,
        output_file=output_file,
        start_date=731947,  # example: 731947 for 2005-01-01
    )


Processing basin: BARENTS
Little check for alignment:
Profile 100: time=2007-08-01 00:00:00, month=8
Profile 101: time=2007-08-01 00:00:00, month=8
Profile 102: time=2007-08-01 00:00:00, month=8
Profile 103: time=2007-08-01 00:00:00, month=8
Profile 104: time=2007-08-01 00:00:00, month=8
Profile 105: time=2007-08-01 00:00:00, month=8
Profile 106: time=2007-08-01 00:00:00, month=8
Profile 107: time=2007-08-01 00:00:00, month=8
Profile 108: time=2007-08-02 00:00:00, month=8
Profile 109: time=2007-08-02 00:00:00, month=8
Profile 110: time=2007-08-02 00:00:00, month=8
Profile 111: time=2007-08-02 00:00:00, month=8
Profile 112: time=2007-08-06 00:00:00, month=8
Profile 113: time=2007-08-08 00:00:00, month=8
Profile 114: time=2008-06-30 00:00:00, month=6
Profile 115: time=2008-06-30 00:00:00, month=6
Profile 116: time=2008-06-30 00:00:00, month=6
Profile 117: time=2008-07-06 00:00:00, month=7
Profile 118: time=2008-07-06 00:00:00, month=7
Profile 119: time=2008-07-07 00:00:00, month=7
Profi

In [5]:
profiles_barents

<xarray.Dataset>
Dimensions:              (profile: 440, depth: 187)
Coordinates:
  * profile              (profile) int64 5615 5663 5664 ... 358423 358424 358445
  * depth                (depth) float64 1.0 3.0 5.0 ... 5.3e+03 5.4e+03 5.5e+03
    lon                  (profile) float64 61.32 61.99 60.8 ... 91.59 91.68
    lat                  (profile) float64 86.77 82.25 82.5 ... 78.36 78.29
Data variables: (12/15)
    longitude            (profile) float64 61.32 61.99 60.8 ... 91.59 91.68
    latitude             (profile) float64 86.77 82.25 82.5 ... 78.36 78.29
    time                 (profile) float64 7.329e+05 7.329e+05 ... 7.351e+05
    source               (profile) <U9 'UDASH' 'UDASH' 'UDASH' ... 'WOD' 'WOD'
    dataset_index        (profile) float64 8.74e+03 9.031e+03 ... 4.9e+04
    is_duplicate         (profile) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    ...                   ...
    nb_depth             (profile) float64 19.0 209.0 299.0 ... 31.0 38.0 27.0
    prof_descr           (profile) object b'akfed_hel_10_S_spis' ... b'ZZ000033'
    temperature_QC       (profile, depth) float64 nan nan -1.56 ... nan nan nan
    temperature_QC_anom  (profile, depth) float64 nan nan 0.1641 ... nan nan nan
    temperature_clim     (profile, depth) float64 -1.727 -1.727 ... nan nan
    month_used_for_clim  (profile) int64 8 8 8 8 8 8 8 8 8 ... 7 8 8 8 8 8 8 8 8
Attributes:
    title:        ARGO temperature profiles
    source:       ARGO
    featureType:  profile
    Comments:     ARGO temperature profiles interpolated on ISAS vertical lev...

In [6]:

dt_list = [datetime.fromordinal(int(t)) for t in profiles_barents.time]